# All models over 10 splits, significance tests and cost (A17)

Every model on the 10 stratified group splits in `sp.SPLIT_SEEDS`, on one machine, with one model seed (42) per fit,
as in `08`: logistic regression (`sp.fit_lr`), XGBoost and LightGBM (parameters from `02`), Random Forest
(configuration from `13`), and the cross-ensemble (XGBoost weight chosen on validation per split,
`sp.select_blend_weight`). Thresholds come from validation.

**Tests.** Per-split differences against LightGBM (and XGBoost against the ensemble) on F1, AP and AUROC: mean, SD,
wins; the **corrected resampled t-test** (Nadeau and Bengio 2003), the headline test because the splits share
training data, `t = mean(d) / sqrt((1/n + n_test/n_train) · var(d))`, `n = 10`, 9 degrees of freedom, with the
unique-row ratio `n_test / n_train` averaged over splits; also the paired t-test and the Wilcoxon signed-rank test,
labeled uncorrected. Holm correction across the comparisons on each metric.

**Cost** (seed-42 split): training wall time per model (single seed: the median of the fits with seeds 42, 123, 456;
3-seed ensemble: their sum), scoring rate on the 130,435 test rows, peak memory increase during fit, pickled size,
and `sp.build_master_df` wall time. The feature-extraction SQL (Flipside, BigQuery; `data/*/readme`) is not timed.

**Results**: `results/17_models_10_splits.json`.

## Setup

In [1]:
import os, pickle, platform, threading, time, warnings
import numpy as np
import pandas as pd
import psutil
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
import sybil_pipeline as sp
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
assert not sp.CODE_COMMIT.endswith('-dirty'), f'Run from a clean working tree (CODE_COMMIT={sp.CODE_COMMIT})'
print('Code commit:', sp.CODE_COMMIT)
PLATFORM = dict(machine=platform.machine(), processor=platform.processor(), system=platform.system(),
                python=platform.python_version(), cpu_count=os.cpu_count(), n_jobs=sp.N_JOBS)
print('Platform:', PLATFORM)

prov = sp.provenance(['02_hyperparameter_search', '08_ablation_families', '13_random_forest_sybil'])
print(prov.to_string(index=False))
assert prov['dependencies_unchanged'].all()

DATA_DIR = './data'
FEATS = sp.FEATS
XGB_P, LGBM_P = sp.load_selected_params()
r13 = sp.load_results('13_random_forest_sybil')
RF_P = {k: v for k, v in r13['params'].items()}
print('XGBoost:', XGB_P); print('LightGBM:', LGBM_P); print('Random Forest (from 13):', RF_P)
t0 = time.time()
df, _, _ = sp.build_master_df(DATA_DIR, verbose=False)
BUILD_SECONDS = time.time() - t0
print(f'build_master_df: {BUILD_SECONDS:.1f}s')
MODELS = ['Logistic regression', 'XGBoost', 'LightGBM', 'Random Forest', 'Ensemble']

Code commit: 2f50786
Platform: {'machine': 'x86_64', 'processor': 'x86_64', 'system': 'Linux', 'python': '3.11.15', 'cpu_count': 4, 'n_jobs': 4}
                notebook  commit  dependencies_unchanged
02_hyperparameter_search 03cea83                    True
    08_ablation_families deef156                    True
  13_random_forest_sybil 2adac1c                    True


XGBoost: {'learning_rate': 0.05, 'max_depth': 12, 'subsample': 0.5, 'colsample_bytree': 0.8, 'min_child_weight': 10}
LightGBM: {'num_leaves': 511, 'learning_rate': 0.05, 'subsample': 1.0, 'reg_lambda': 1.0}
Random Forest (from 13): {'max_features': 0.3, 'min_samples_leaf': 1, 'max_depth': None, 'max_samples': None, 'n_estimators': 300, 'criterion': 'gini', 'bootstrap': True, 'n_jobs': 4}


build_master_df: 60.3s


## Per split

In [2]:
def rf_fit(S, seed=42):
    m = RandomForestClassifier(**RF_P, random_state=seed)
    m.fit(S['X_train'], S['y_train'])
    return dict(val=m.predict_proba(S['X_val'])[:, 1], test=m.predict_proba(S['X_test'])[:, 1], model=m)

rows, cats, extra = [], [], []
for s in sp.SPLIT_SEEDS:
    t0 = time.time()
    S = sp.make_splits(df, FEATS, method='group', seed=s)
    cat = df.loc[S['idx_test'], 'category']
    P = {'Logistic regression': sp.fit_lr(S),
         'XGBoost': sp.fit_xgb(S, XGB_P, seeds=(42,), verbose=False),
         'LightGBM': sp.fit_lgbm(S, LGBM_P, seeds=(42,), verbose=False),
         'Random Forest': rf_fit(S)}
    w, _ = sp.select_blend_weight(S['y_val'], P['XGBoost']['val'], P['LightGBM']['val'])
    P['Ensemble'] = {part: w * P['XGBoost'][part] + (1 - w) * P['LightGBM'][part] for part in ['val', 'test']}
    for name in MODELS:
        r = sp.evaluate(name, S['y_val'], P[name]['val'], S['y_test'], P[name]['test'])
        rows.append(dict(split_seed=s, model=name, **{k: float(r[k]) for k in ['threshold', 'precision', 'recall', 'f1', 'auroc', 'ap']}))
        bc = sp.metrics_by_category(S['y_test'], P[name]['test'], cat, r['threshold'])
        assert bc['n'].sum() == len(S['y_test']) and bc['sybil'].sum() == S['y_test'].sum()
        cats += [dict(split_seed=s, model=name, **x) for x in bc.to_dict('records')]
    extra.append(dict(split_seed=s, xgb_weight=w, xgb_rounds=P['XGBoost']['rounds'][0], lgbm_rounds=P['LightGBM']['rounds'][0],
                      n_train_unique=len(S['idx_train']), n_val=len(S['idx_val']), n_test=len(S['idx_test'])))
    f1 = {r['model']: r['f1'] for r in rows if r['split_seed'] == s}
    print(f"split {s}: " + ', '.join(f'{k} {v:.4f}' for k, v in f1.items()) + f'; w={w:.2f} ({time.time() - t0:.0f}s)', flush=True)
per_split = pd.DataFrame(rows)
by_category = pd.DataFrame(cats)
extra = pd.DataFrame(extra)
for m in MODELS:
    assert (per_split['model'] == m).sum() == 10 and per_split.loc[per_split['model'] == m, 'split_seed'].is_unique

split 42: Logistic regression 0.2400, XGBoost 0.7148, LightGBM 0.7175, Random Forest 0.7112, Ensemble 0.7176; w=0.50 (461s)


split 1: Logistic regression 0.2321, XGBoost 0.6958, LightGBM 0.6924, Random Forest 0.6992, Ensemble 0.6959; w=0.30 (486s)


split 2: Logistic regression 0.2397, XGBoost 0.7047, LightGBM 0.7142, Random Forest 0.7010, Ensemble 0.7099; w=0.62 (492s)


split 3: Logistic regression 0.2348, XGBoost 0.7114, LightGBM 0.7109, Random Forest 0.6932, Ensemble 0.7141; w=0.76 (450s)


split 4: Logistic regression 0.2344, XGBoost 0.7160, LightGBM 0.7076, Random Forest 0.7023, Ensemble 0.7165; w=0.60 (453s)


split 5: Logistic regression 0.2373, XGBoost 0.7077, LightGBM 0.7126, Random Forest 0.6993, Ensemble 0.7138; w=0.46 (480s)


split 6: Logistic regression 0.2320, XGBoost 0.7000, LightGBM 0.6971, Random Forest 0.6992, Ensemble 0.6981; w=0.32 (481s)


split 7: Logistic regression 0.2326, XGBoost 0.7093, LightGBM 0.7106, Random Forest 0.7084, Ensemble 0.7128; w=0.58 (459s)


split 8: Logistic regression 0.2310, XGBoost 0.7145, LightGBM 0.7177, Random Forest 0.7056, Ensemble 0.7186; w=0.76 (459s)


split 9: Logistic regression 0.2288, XGBoost 0.7011, LightGBM 0.7053, Random Forest 0.7041, Ensemble 0.7025; w=0.34 (500s)


### Check against `08` (LightGBM, split seed 42)

In [3]:
r08 = pd.DataFrame(sp.load_results('08_ablation_families')['per_split'])
r08 = r08[r08['config'] == 'All features'].set_index('split_seed')
mine = per_split[per_split['model'] == 'LightGBM'].set_index('split_seed')
d08 = (mine['f1'] - r08['test_f1']).abs()
LGBM_MATCHES_08 = bool((d08 < 1e-12).all())
print(f"LightGBM here vs 08 (test F1): max |difference| {d08.max():.4f} over 10 splits; identical: {LGBM_MATCHES_08}")
if not LGBM_MATCHES_08:
    print(f"  08 was produced on another machine; this notebook ran on {PLATFORM['machine']} / {PLATFORM['system']}.")

LightGBM here vs 08 (test F1): max |difference| 0.0000 over 10 splits; identical: True


## Summary and significance tests

In [4]:
summary = {}
for m in MODELS:
    t = per_split[per_split['model'] == m]
    summary[m] = {f'{k}_{a}': float(getattr(t[k], a)()) for k in ['precision', 'recall', 'f1', 'ap', 'auroc'] for a in ('mean', 'std')}
RATIO = float((extra['n_test'] / extra['n_train_unique']).mean())
N = 10

def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); out = np.empty_like(p); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (len(p) - rank) * p[i])); out[i] = run
    return out

PAIRS = [(m, 'LightGBM') for m in ['Logistic regression', 'XGBoost', 'Random Forest', 'Ensemble']] + [('XGBoost', 'Ensemble')]
tests = []
for metric in ['f1', 'ap', 'auroc']:
    block = []
    for a, b in PAIRS:
        A = per_split[per_split['model'] == a].set_index('split_seed')[metric]
        B = per_split[per_split['model'] == b].set_index('split_seed')[metric]
        d = (A - B).loc[sp.SPLIT_SEEDS].to_numpy()
        var = d.var(ddof=1)
        t_c = d.mean() / np.sqrt((1 / N + RATIO) * var) if var > 0 else np.nan
        p_c = float(2 * stats.t.sf(abs(t_c), N - 1)) if var > 0 else np.nan
        tt = stats.ttest_rel(A.loc[sp.SPLIT_SEEDS], B.loc[sp.SPLIT_SEEDS])
        wx = stats.wilcoxon(d) if np.any(d != 0) else None
        block.append(dict(metric=metric, model=a, against=b, mean_diff=float(d.mean()), sd_diff=float(d.std(ddof=1)),
                          higher_n=int((d > 0).sum()), t_corrected=float(t_c), p_corrected=p_c,
                          t_paired_uncorrected=float(tt.statistic), p_paired_uncorrected=float(tt.pvalue),
                          p_wilcoxon_uncorrected=float(wx.pvalue) if wx else np.nan))
    for k in ['p_corrected', 'p_paired_uncorrected', 'p_wilcoxon_uncorrected']:
        adj = holm([x[k] for x in block])
        for x, v in zip(block, adj):
            x[f'{k}_holm'] = float(v)
    tests += block
tests = pd.DataFrame(tests)
print(f'n_test / n_train (unique rows, mean over splits) = {RATIO:.4f}')
print(tests.round(4).to_string(index=False))

n_test / n_train (unique rows, mean over splits) = 0.6122
metric               model  against  mean_diff  sd_diff  higher_n  t_corrected  p_corrected  t_paired_uncorrected  p_paired_uncorrected  p_wilcoxon_uncorrected  p_corrected_holm  p_paired_uncorrected_holm  p_wilcoxon_uncorrected_holm
    f1 Logistic regression LightGBM    -0.4743   0.0072         0     -78.1633       0.0000             -208.6009                0.0000                  0.0020            0.0000                     0.0000                       0.0098
    f1             XGBoost LightGBM    -0.0010   0.0050         4      -0.2433       0.8132               -0.6494                0.5323                  0.4922            1.0000                     0.5323                       0.4922
    f1       Random Forest LightGBM    -0.0062   0.0078         2      -0.9501       0.3669               -2.5356                0.0319                  0.0488            1.0000                     0.0958                       0.1465
    f1

### Ready-to-cite table (10 group splits, model seed 42; p: corrected resampled t-test vs LightGBM, Holm-adjusted)

In [5]:
cite = []
for m in MODELS:
    row = dict(model=m)
    for k, lab in [('f1', 'F1'), ('ap', 'AP'), ('auroc', 'AUROC')]:
        row[lab] = f"{summary[m][f'{k}_mean']:.4f} ± {summary[m][f'{k}_std']:.4f}"
        if m != 'LightGBM':
            p = tests[(tests.metric == k) & (tests.model == m) & (tests.against == 'LightGBM')]['p_corrected_holm'].iloc[0]
            row[f'p {lab}'] = f'{p:.3g}'
        else:
            row[f'p {lab}'] = '—'
    cite.append(row)
cite = pd.DataFrame(cite)
print(cite.to_string(index=False))
print(f"\nBlend weight per split: {extra['xgb_weight'].tolist()}")
print(f"XGBoost rounds: {extra['xgb_rounds'].tolist()}\nLightGBM rounds: {extra['lgbm_rounds'].tolist()}")
cat_summary = (by_category[by_category['category'].isin(['IxL', 'IxI', 'IxE'])]
               .groupby(['model', 'category'])[['precision', 'recall', 'f1', 'auroc']].agg(['mean', 'std']))
print(cat_summary.round(4).to_string())

              model              F1     p F1              AP     p AP           AUROC  p AUROC
Logistic regression 0.2343 ± 0.0037 2.32e-13 0.1626 ± 0.0079 6.39e-13 0.8337 ± 0.0026 2.11e-12
            XGBoost 0.7075 ± 0.0069        1 0.7545 ± 0.0068    0.234 0.9688 ± 0.0012        1
           LightGBM 0.7086 ± 0.0083        — 0.7606 ± 0.0071        — 0.9688 ± 0.0019        —
      Random Forest 0.7023 ± 0.0052        1 0.7537 ± 0.0077    0.234 0.9689 ± 0.0014        1
           Ensemble 0.7100 ± 0.0082        1 0.7627 ± 0.0064    0.351 0.9699 ± 0.0016   0.0425

Blend weight per split: [0.5, 0.3, 0.62, 0.76, 0.6, 0.46, 0.32, 0.58, 0.76, 0.34]
XGBoost rounds: [1214, 1262, 1324, 1162, 1054, 1378, 1209, 1235, 1268, 1189]
LightGBM rounds: [315, 292, 332, 299, 275, 316, 297, 327, 322, 363]


                             precision          recall              f1           auroc        
                                  mean     std    mean     std    mean     std    mean     std
model               category                                                                  
Ensemble            IxE         0.6946  0.1106  0.1457  0.0367  0.2398  0.0543  0.9204  0.0172
                    IxI         0.8927  0.0718  0.2740  0.1091  0.4061  0.1309  0.9266  0.0103
                    IxL         0.7090  0.0164  0.7747  0.0197  0.7401  0.0054  0.9762  0.0013
LightGBM            IxE         0.7018  0.1353  0.1348  0.0490  0.2239  0.0733  0.9177  0.0166
                    IxI         0.8544  0.1337  0.2778  0.1115  0.4025  0.1318  0.9229  0.0127
                    IxL         0.7209  0.0193  0.7613  0.0267  0.7399  0.0052  0.9757  0.0013
Logistic regression IxE         0.1170  0.0496  0.2107  0.0873  0.1476  0.0593  0.8126  0.0342
                    IxI         0.1943  0.0183  0.

## Cost (seed-42 split)

Each model is trained with seeds 42, 123 and 456 (logistic regression has no seed: three repeated fits). Single-seed
time is the median of the three fits; the 3-seed ensemble time is their sum. Peak memory is the largest increase in
process RSS over the level before the fit, sampled every 20 ms in a background thread. Scoring is `predict_proba` on
the 130,435 test rows (median of 3).

In [6]:
class PeakRSS:
    def __init__(self):
        self.proc = psutil.Process(); self.peak = 0; self.stop = False
    def __enter__(self):
        self.base = self.proc.memory_info().rss; self.peak = self.base
        self.th = threading.Thread(target=self._run, daemon=True); self.th.start(); return self
    def _run(self):
        while not self.stop:
            self.peak = max(self.peak, self.proc.memory_info().rss); time.sleep(0.02)
    def __exit__(self, *a):
        self.stop = True; self.th.join(); self.peak = max(self.peak, self.proc.memory_info().rss)
    @property
    def increase(self):
        return self.peak - self.base

S = sp.make_splits(df, FEATS, method='group', seed=sp.SEED)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb

def fit_one(name, seed):
    if name == 'Logistic regression':
        m = Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(**sp.LR_PARAMS))]); m.fit(S['X_train'], S['y_train'])
    elif name == 'XGBoost':
        m = sp.xgb_model(XGB_P, seed); m.fit(S['X_train'], S['y_train'], eval_set=[(S['X_val'], S['y_val'])], verbose=False)
    elif name == 'LightGBM':
        m = sp.lgbm_model(LGBM_P, seed)
        m.fit(S['X_train'], S['y_train'], eval_set=[(S['X_val'], S['y_val'])],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    else:
        m = RandomForestClassifier(**RF_P, random_state=seed); m.fit(S['X_train'], S['y_train'])
    return m

cost = []
for name in ['Logistic regression', 'XGBoost', 'LightGBM', 'Random Forest']:
    fits, mems, score_t, sizes = [], [], [], []
    for seed in sp.MODEL_SEEDS:
        with PeakRSS() as pk:
            t0 = time.perf_counter(); m = fit_one(name, seed); fits.append(time.perf_counter() - t0)
        mems.append(pk.increase)
        st = []
        for _ in range(3):
            t0 = time.perf_counter(); m.predict_proba(S['X_test']); st.append(time.perf_counter() - t0)
        score_t.append(float(np.median(st)))
        sizes.append(len(pickle.dumps(m)))
        del m
    cost.append(dict(model=name, train_seconds_single_median=float(np.median(fits)), train_seconds_3seed=float(np.sum(fits)),
                     train_seconds_each=[round(x, 2) for x in fits],
                     score_seconds_test_median=float(np.median(score_t)),
                     score_rows_per_second=float(len(S['X_test']) / np.median(score_t)),
                     peak_rss_increase_mib_median=float(np.median(mems) / 2**20),
                     pickled_mib_median=float(np.median(sizes) / 2**20)))
    print(f"{name}: train {np.median(fits):.1f}s (3 seeds {np.sum(fits):.1f}s), score {len(S['X_test']) / np.median(score_t):,.0f} rows/s, "
          f"peak +{np.median(mems) / 2**20:,.0f} MiB, pickle {np.median(sizes) / 2**20:,.1f} MiB", flush=True)
cost = pd.DataFrame(cost)
print(cost.drop(columns='train_seconds_each').round(2).to_string(index=False))
print(f'build_master_df: {BUILD_SECONDS:.1f}s (feature-extraction SQL not timed)')

Logistic regression: train 37.4s (3 seeds 110.9s), score 1,311,451 rows/s, peak +984 MiB, pickle 0.0 MiB


XGBoost: train 71.3s (3 seeds 215.5s), score 60,879 rows/s, peak +121 MiB, pickle 12.7 MiB


LightGBM: train 40.9s (3 seeds 122.2s), score 40,716 rows/s, peak +97 MiB, pickle 16.0 MiB


Random Forest: train 290.0s (3 seeds 865.3s), score 59,962 rows/s, peak +247 MiB, pickle 268.8 MiB


              model  train_seconds_single_median  train_seconds_3seed  score_seconds_test_median  score_rows_per_second  peak_rss_increase_mib_median  pickled_mib_median
Logistic regression                        37.44               110.94                       0.10             1311451.21                        984.24                0.00
            XGBoost                        71.31               215.46                       2.14               60878.54                        121.35               12.65
           LightGBM                        40.86               122.22                       3.20               40715.68                         96.56               16.03
      Random Forest                       290.03               865.34                       2.18               59962.34                        246.78              268.82
build_master_df: 60.3s (feature-extraction SQL not timed)


## Save results

In [7]:
sp.save_results([], '17_models_10_splits', extra=dict(
    params=dict(xgb=XGB_P, lgbm=LGBM_P, rf=RF_P, lr=sp.LR_PARAMS), model_seed=42,
    per_split=per_split.to_dict('records'), by_category=by_category.to_dict('records'),
    split_info=extra.to_dict('records'), summary=summary, tests=tests.to_dict('records'),
    test_method=dict(headline='corrected resampled t-test (Nadeau and Bengio 2003), df = 9, Holm across comparisons per metric',
                     n_test_over_n_train=RATIO, also='paired t-test and Wilcoxon signed-rank, uncorrected'),
    cite_table=cite.to_dict('records'), lgbm_matches_08=LGBM_MATCHES_08,
    cost=dict(table=cost.to_dict('records'), build_master_df_seconds=BUILD_SECONDS, test_rows=len(S['X_test']),
              note='single-seed time is the median of the fits with seeds 42, 123, 456; 3-seed time is their sum. '
                   'The feature-extraction SQL (Flipside and BigQuery; data/*/readme) is not timed here.'),
    platform=PLATFORM))

Saved results/17_models_10_splits.json
